# 02 — Data Cleaning & Preprocessing

## Objective
Clean all four raw tables while preserving the time-series
structure of weekly activity data and correctly distinguishing
between conditionally expected nulls versus genuine data
quality problems.

## What This Notebook Covers
- Removing exact duplicate rows
- Fixing data types
- Classifying nulls: conditionally expected vs quality problems
- Handling quality nulls with appropriate strategies
- Fixing corrupt values (ages, impossible rates)
- Outlier handling specific to time series data
- Verifying churn label integrity after all cleaning
- Saving cleaned tables to data/processed/

## The New Concept: Conditionally Expected Nulls

A null is "conditionally expected" when the value is
correctly absent given another column's value.

Example in this dataset:
  weekly_rating is null when rides_this_week = 0
  → Correct — you cannot rate a driver with no rides
  → Do NOT fill this with a median
  → Fill with forward-fill within the driver's series
    (carry forward the last known rating)

Example of quality null:
  weekly_earnings is null despite rides_this_week > 0
  → This is a data collection failure
  → Safe to fill with median earnings for that vehicle type

Getting this distinction wrong corrupts the trend analysis
that is central to churn prediction.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

plt.rcParams["figure.dpi"]        = 130
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

CHURN_COLORS = {0: "#4CAF50", 1: "#F44336"}
CHURN_LABELS = {0: "Retained", 1: "Churned"}

print("Libraries loaded ✅")

Libraries loaded ✅


In [2]:
DATA_RAW       = "../data/raw/"
DATA_PROCESSED = "../data/processed/"
os.makedirs(DATA_PROCESSED, exist_ok=True)

drivers_df   = pd.read_csv(DATA_RAW + "drivers.csv",
                            parse_dates=["join_date"])
weekly_df    = pd.read_csv(DATA_RAW + "weekly_activity.csv",
                            parse_dates=["week_start_date"])
incentive_df = pd.read_csv(DATA_RAW + "incentives.csv")
tickets_df   = pd.read_csv(DATA_RAW + "support_tickets.csv")

print("Raw data loaded ✅")
print(f"  drivers          : {drivers_df.shape}")
print(f"  weekly_activity  : {weekly_df.shape}")
print(f"  incentives       : {incentive_df.shape}")
print(f"  support_tickets  : {tickets_df.shape}")

print(f"\nChurn rate before cleaning: "
      f"{drivers_df['is_churned'].mean()*100:.3f}%")
print("(this must stay stable through all cleaning steps)")

Raw data loaded ✅
  drivers          : (5000, 20)
  weekly_activity  : (80000, 17)
  incentives       : (7319, 6)
  support_tickets  : (10030, 6)

Churn rate before cleaning: 40.300%
(this must stay stable through all cleaning steps)


## Step 1 — Remove Duplicate Rows

### New Consideration for Weekly Activity
weekly_activity has a natural composite key:
(driver_id, week_number) — one row per driver per week.

Duplicate detection must use this composite key,
not just driver_id or week_number alone.
A single driver_id appearing twice is expected
(they have 16 rows — one per week).
The same (driver_id, week_number) pair appearing twice
is a system glitch duplicate.

### Churn Rate Monitoring
After every deduplication step we verify the churn
rate in drivers_df has not shifted. Any cleaning step
that removes churn labels disproportionately from
one class corrupts the modeling dataset.

In [3]:
before_churn_rate = drivers_df["is_churned"].mean() * 100

before = {name: len(df) for name, df in [
    ("drivers",          drivers_df),
    ("weekly_activity",  weekly_df),
    ("incentives",       incentive_df),
    ("support_tickets",  tickets_df),
]}

# ── Exact duplicate rows ──────────────────────────────────────
drivers_df   = drivers_df.drop_duplicates().reset_index(drop=True)
weekly_df    = weekly_df.drop_duplicates().reset_index(drop=True)
incentive_df = incentive_df.drop_duplicates().reset_index(drop=True)
tickets_df   = tickets_df.drop_duplicates().reset_index(drop=True)

# ── Primary key duplicates ────────────────────────────────────
drivers_df = drivers_df.drop_duplicates(
    subset=["driver_id"]
).reset_index(drop=True)

# Weekly: composite key (driver_id, week_number)
weekly_df = weekly_df.drop_duplicates(
    subset=["driver_id", "week_number"]
).reset_index(drop=True)

incentive_df = incentive_df.drop_duplicates().reset_index(drop=True)
tickets_df   = tickets_df.drop_duplicates().reset_index(drop=True)

after_churn_rate = drivers_df["is_churned"].mean() * 100

after = {name: len(df) for name, df in [
    ("drivers",         drivers_df),
    ("weekly_activity", weekly_df),
    ("incentives",      incentive_df),
    ("support_tickets", tickets_df),
]}

print("Duplicate Removal Summary")
print(f"{'Table':<20} {'Before':>8} {'After':>8} {'Removed':>8}")
print("-" * 50)
for name in before:
    removed = before[name] - after[name]
    print(f"{name:<20} {before[name]:>8,} "
          f"{after[name]:>8,} {removed:>8,}")

print(f"\n✅ Churn rate check:")
print(f"   Before: {before_churn_rate:.3f}%")
print(f"   After : {after_churn_rate:.3f}%")
print(f"   Drift : {after_churn_rate - before_churn_rate:+.4f}%")

Duplicate Removal Summary
Table                  Before    After  Removed
--------------------------------------------------
drivers                 5,000    5,000        0
weekly_activity        80,000   80,000        0
incentives              7,319    7,283       36
support_tickets        10,030   10,028        2

✅ Churn rate check:
   Before: 40.300%
   After : 40.300%
   Drift : +0.0000%


In [4]:
# ── drivers ──────────────────────────────────────────────────
drivers_df["join_date"]   = pd.to_datetime(
    drivers_df["join_date"], errors="coerce"
)
for col in ["age","days_on_platform","experience_years",
            "zones_known","rating"]:
    if col in drivers_df.columns:
        drivers_df[col] = pd.to_numeric(
            drivers_df[col], errors="coerce"
        )
for col in ["has_other_income","uses_competitor_app",
            "bank_account_linked","is_prime_driver"]:
    if col in drivers_df.columns:
        drivers_df[col] = drivers_df[col].astype(bool)

drivers_df["is_churned"]    = drivers_df["is_churned"].astype(int)
drivers_df["churn_profile"] = drivers_df["churn_profile"].astype(str)

# ── weekly_activity ───────────────────────────────────────────
weekly_df["week_start_date"] = pd.to_datetime(
    weekly_df["week_start_date"], errors="coerce"
)
for col in ["rides_this_week","hours_online","cancellation_rate",
            "weekly_earnings","surge_factor","weekly_rating",
            "incentive_amount","complaints","zones_covered",
            "login_days","week_number","month"]:
    if col in weekly_df.columns:
        weekly_df[col] = pd.to_numeric(
            weekly_df[col], errors="coerce"
        )
weekly_df["incentive_received"] = weekly_df[
    "incentive_received"
].astype(bool)
weekly_df["will_churn"]  = weekly_df["will_churn"].astype(bool)

# ── incentives ────────────────────────────────────────────────
for col in ["offer_value","rides_increase"]:
    incentive_df[col] = pd.to_numeric(
        incentive_df[col], errors="coerce"
    )
incentive_df["was_accepted"] = incentive_df[
    "was_accepted"
].astype(bool)

# ── support_tickets ───────────────────────────────────────────
for col in ["resolution_days","satisfaction"]:
    tickets_df[col] = pd.to_numeric(
        tickets_df[col], errors="coerce"
    )

print("Data types fixed ✅")
print("\nKey dtype verification:")
print(f"  is_churned          : {drivers_df['is_churned'].dtype}")
print(f"  rides_this_week     : {weekly_df['rides_this_week'].dtype}")
print(f"  week_start_date     : {weekly_df['week_start_date'].dtype}")
print(f"  incentive_received  : {weekly_df['incentive_received'].dtype}")

Data types fixed ✅

Key dtype verification:
  is_churned          : int64
  rides_this_week     : int64
  week_start_date     : datetime64[us]
  incentive_received  : bool


## Step 3 — Classify Every Null Before Touching Anything

This is the most important analytical step in this notebook.
Every null column gets classified into one of three categories:

### Category A — Conditionally Expected Nulls
These nulls are CORRECT given another column's value.
Strategy: Forward-fill WITHIN driver, not global median.

| Column | Condition | Why Forward-Fill |
|---|---|---|
| weekly_rating | rides_this_week = 0 | No rides = no rating. Carry last known rating. |
| weekly_earnings | rides_this_week = 0 | No rides = no earnings. Already 0 by design, null = data issue |

### Category B — Structural Zeros Stored as Null
These should be 0.0 but arrived as null due to system issue.

| Column | Why It Should Be Zero |
|---|---|
| incentive_amount | When incentive_received = False, amount should be 0 |
| complaints | When rides = 0, complaints should be 0 |
| zones_covered | When rides = 0, zones should be 0 |

### Category C — Pure Data Quality Nulls
Random missing values with no relationship to other columns.
Strategy: Standard imputation.

| Column | Strategy |
|---|---|
| hours_online | Median by vehicle_type and churn status |
| cancellation_rate | Median by churn status |
| weekly_earnings (when rides > 0) | Median by vehicle_type |
| driver rating | Median |
| driver age | Median after corrupt removal |

### What We Do NOT Impute
- driver phone, referral_source — cannot fabricate contact info
- support_ticket satisfaction when pending — null is correct
  (pending tickets have no satisfaction score yet)
- support_ticket resolution_days when pending — not resolved yet

In [5]:
print("NULL AUDIT — WEEKLY ACTIVITY TABLE\n")

# For each null column, check if nulls correlate
# with rides_this_week == 0
null_cols_weekly = [
    c for c in weekly_df.columns
    if weekly_df[c].isnull().sum() > 0
]

for col in null_cols_weekly:
    total_null = weekly_df[col].isnull().sum()
    pct_null   = total_null / len(weekly_df) * 100

    # How many nulls coincide with zero rides?
    null_when_zero = (
        weekly_df[col].isnull() &
        (weekly_df["rides_this_week"] == 0)
    ).sum()
    pct_zero_rides = (
        null_when_zero / total_null * 100
        if total_null > 0 else 0
    )

    category = (
        "✅ Conditionally expected"
        if pct_zero_rides > 70
        else "⚠️ Data quality null"
    )

    print(f"  {col:<25} "
          f"null={total_null:>5,} ({pct_null:.1f}%)  "
          f"zero-rides={null_when_zero:>5,} "
          f"({pct_zero_rides:.1f}%)  {category}")

print(f"\nNULL AUDIT — DRIVERS TABLE")
driver_nulls = drivers_df.isnull().sum()
driver_nulls = driver_nulls[driver_nulls > 0]
for col, count in driver_nulls.items():
    pct = count / len(drivers_df) * 100
    print(f"  {col:<25} null={count:>4,} ({pct:.2f}%)")

print(f"\nNULL AUDIT — SUPPORT TICKETS TABLE")
ticket_nulls = tickets_df.isnull().sum()
ticket_nulls = ticket_nulls[ticket_nulls > 0]
for col, count in ticket_nulls.items():
    pct = count / len(tickets_df) * 100
    print(f"  {col:<25} null={count:>4,} ({pct:.2f}%)")

NULL AUDIT — WEEKLY ACTIVITY TABLE

  hours_online              null=2,048 (2.6%)  zero-rides=  327 (16.0%)  ⚠️ Data quality null
  cancellation_rate         null=1,165 (1.5%)  zero-rides=  169 (14.5%)  ⚠️ Data quality null
  weekly_earnings           null=1,583 (2.0%)  zero-rides=  256 (16.2%)  ⚠️ Data quality null
  weekly_rating             null=14,746 (18.4%)  zero-rides=12,324 (83.6%)  ✅ Conditionally expected
  churn_reason              null=47,760 (59.7%)  zero-rides=    0 (0.0%)  ⚠️ Data quality null

NULL AUDIT — DRIVERS TABLE
  phone                     null=  96 (1.92%)
  vehicle_year              null= 562 (11.24%)
  rating                    null= 176 (3.52%)
  referral_source           null= 962 (19.24%)

NULL AUDIT — SUPPORT TICKETS TABLE
  resolution_days           null=2,818 (28.10%)
  satisfaction              null=2,022 (20.16%)


In [6]:
print("Fixing structural zeros (nulls that should be 0)...\n")

# When incentive_received is False, amount should be 0
mask_no_incentive = ~weekly_df["incentive_received"]
null_incentive    = (
    weekly_df["incentive_amount"].isnull() & mask_no_incentive
).sum()
weekly_df.loc[
    mask_no_incentive & weekly_df["incentive_amount"].isnull(),
    "incentive_amount"
] = 0.0
print(f"  incentive_amount: {null_incentive:,} nulls → 0.0 "
      f"(no incentive received)")

# When rides = 0, complaints and zones should be 0
for col in ["complaints", "zones_covered"]:
    mask_zero_rides = weekly_df["rides_this_week"] == 0
    null_count = (
        weekly_df[col].isnull() & mask_zero_rides
    ).sum()
    weekly_df.loc[
        mask_zero_rides & weekly_df[col].isnull(), col
    ] = 0
    print(f"  {col}: {null_count:,} nulls → 0 (zero-ride weeks)")

print("\n✅ Structural zeros fixed")

Fixing structural zeros (nulls that should be 0)...

  incentive_amount: 0 nulls → 0.0 (no incentive received)
  complaints: 0 nulls → 0 (zero-ride weeks)
  zones_covered: 0 nulls → 0 (zero-ride weeks)

✅ Structural zeros fixed


In [7]:
print("Forward-filling conditionally expected nulls "
      "within driver time series...\n")

# Sort by driver and week before forward fill
weekly_df = weekly_df.sort_values(
    ["driver_id", "week_number"]
).reset_index(drop=True)

# weekly_rating — forward fill within driver
# Logic: if driver had rating 4.2 in week 3 and zero rides
# in week 4, their "current rating" is still 4.2
before_null = weekly_df["weekly_rating"].isnull().sum()

weekly_df["weekly_rating"] = weekly_df.groupby(
    "driver_id"
)["weekly_rating"].transform(
    lambda x: x.ffill().bfill()
)

after_null = weekly_df["weekly_rating"].isnull().sum()

print(f"  weekly_rating:")
print(f"    Before forward-fill: {before_null:,} nulls")
print(f"    After forward-fill : {after_null:,} nulls")
if after_null > 0:
    # Any remaining nulls = driver never had a rating
    # Fill with overall median
    median_rating = weekly_df["weekly_rating"].median()
    weekly_df["weekly_rating"] = weekly_df[
        "weekly_rating"
    ].fillna(median_rating)
    print(f"    Remaining filled with median: {median_rating:.3f}")

print(f"\n✅ Forward-fill complete")

Forward-filling conditionally expected nulls within driver time series...

  weekly_rating:
    Before forward-fill: 14,746 nulls
    After forward-fill : 2,816 nulls
    Remaining filled with median: 4.105

✅ Forward-fill complete


In [8]:
print("Handling data quality nulls...\n")

# ── weekly_earnings (when rides > 0 but earnings null) ────────
# These are genuine missing values — rides happened, payment
# wasn't recorded
rides_positive = weekly_df["rides_this_week"] > 0
earnings_null  = weekly_df["weekly_earnings"].isnull()
quality_earn_null = (rides_positive & earnings_null).sum()

# Fill with median earnings per vehicle type
# (we need vehicle_type from drivers table)
vehicle_map = drivers_df.set_index(
    "driver_id"
)["vehicle_type"].to_dict()
weekly_df["vehicle_type_temp"] = weekly_df[
    "driver_id"
].map(vehicle_map)

median_earn_by_vehicle = weekly_df[
    weekly_df["weekly_earnings"].notna()
].groupby("vehicle_type_temp")["weekly_earnings"].median()

def fill_earnings(row):
    if pd.isnull(row["weekly_earnings"]) and row["rides_this_week"] > 0:
        return median_earn_by_vehicle.get(
            row["vehicle_type_temp"],
            weekly_df["weekly_earnings"].median()
        )
    elif pd.isnull(row["weekly_earnings"]):
        return 0.0  # zero rides = zero earnings
    return row["weekly_earnings"]

weekly_df["weekly_earnings"] = weekly_df.apply(
    fill_earnings, axis=1
)
weekly_df = weekly_df.drop(columns=["vehicle_type_temp"])
print(f"  weekly_earnings: {quality_earn_null:,} quality nulls fixed")

# ── hours_online ──────────────────────────────────────────────
hours_null = weekly_df["hours_online"].isnull().sum()
median_hours = weekly_df.groupby(
    weekly_df["rides_this_week"] > 0
)["hours_online"].transform("median")
weekly_df["hours_online"] = weekly_df[
    "hours_online"
].fillna(median_hours)
print(f"  hours_online: {hours_null:,} nulls → "
      f"median by active/inactive")

# ── cancellation_rate ─────────────────────────────────────────
cancel_null = weekly_df["cancellation_rate"].isnull().sum()
median_cancel = weekly_df["cancellation_rate"].median()
weekly_df["cancellation_rate"] = weekly_df[
    "cancellation_rate"
].fillna(median_cancel)
print(f"  cancellation_rate: {cancel_null:,} nulls → "
      f"median ({median_cancel:.3f})")

# ── login_days ────────────────────────────────────────────────
login_null = weekly_df["login_days"].isnull().sum()
weekly_df["login_days"] = weekly_df["login_days"].fillna(0)
print(f"  login_days: {login_null:,} nulls → 0")

# ── drivers table ─────────────────────────────────────────────
corrupt_age = (
    (drivers_df["age"] < 18) | (drivers_df["age"] > 70)
).sum()
drivers_df.loc[drivers_df["age"] < 18, "age"] = np.nan
drivers_df.loc[drivers_df["age"] > 70, "age"] = np.nan
drivers_df["age"] = drivers_df["age"].fillna(
    drivers_df["age"].median()
)
print(f"\n  driver age: {corrupt_age} corrupt values fixed")

drivers_df["rating"] = drivers_df["rating"].fillna(
    drivers_df["rating"].median()
)
drivers_df["zones_known"] = drivers_df["zones_known"].fillna(
    drivers_df["zones_known"].median()
)
drivers_df["referral_source"] = drivers_df[
    "referral_source"
].fillna("unknown")

# ── support tickets ───────────────────────────────────────────
# resolution_days null when ticket is pending — leave as null
# satisfaction null when pending — leave as null
# These will become "has_pending_ticket" features in FE
print(f"  support tickets: pending nulls left as-is (correct)")

print("\n✅ Quality nulls handled")

Handling data quality nulls...

  weekly_earnings: 1,327 quality nulls fixed
  hours_online: 2,048 nulls → median by active/inactive
  cancellation_rate: 1,165 nulls → median (0.149)
  login_days: 0 nulls → 0

  driver age: 0 corrupt values fixed
  support tickets: pending nulls left as-is (correct)

✅ Quality nulls handled


In [9]:
print("OUTLIER HANDLING — Time Series Aware\n")
print("Note: We cap GENTLY here because trend features depend")
print("on relative values across weeks, not absolute caps.\n")

def cap_weekly_col(col, lower_pct=0.01, upper_pct=0.99):
    """Cap at percentiles — gentle outlier treatment."""
    lower = weekly_df[col].quantile(lower_pct)
    upper = weekly_df[col].quantile(upper_pct)
    before_low  = (weekly_df[col] < lower).sum()
    before_high = (weekly_df[col] > upper).sum()
    weekly_df[col] = weekly_df[col].clip(lower=lower, upper=upper)
    print(f"  {col:<25} "
          f"[{lower:.2f}, {upper:.2f}]  "
          f"capped: {before_low+before_high}")
    return lower, upper

print("Weekly activity — 1st/99th percentile capping:")
for col in ["rides_this_week","hours_online",
            "weekly_earnings","cancellation_rate"]:
    cap_weekly_col(col)

print("\nDrivers table — IQR capping:")
for col in ["days_on_platform","zones_known"]:
    Q1  = drivers_df[col].quantile(0.25)
    Q3  = drivers_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = (
        (drivers_df[col] < lower) |
        (drivers_df[col] > upper)
    ).sum()
    drivers_df[col] = drivers_df[col].clip(
        lower=lower, upper=upper
    )
    print(f"  {col:<25} [{lower:.1f}, {upper:.1f}]  "
          f"capped: {before}")

print("\n✅ Outlier capping complete")

OUTLIER HANDLING — Time Series Aware

Note: We cap GENTLY here because trend features depend
on relative values across weeks, not absolute caps.

Weekly activity — 1st/99th percentile capping:
  rides_this_week           [0.00, 63.00]  capped: 733
  hours_online              [0.10, 31.80]  capped: 1065
  weekly_earnings           [0.00, 18506.59]  capped: 800
  cancellation_rate         [0.05, 0.50]  capped: 566

Drivers table — IQR capping:
  days_on_platform          [-723.6, 2213.4]  capped: 0
  zones_known               [-10.0, 38.0]  capped: 0

✅ Outlier capping complete


In [10]:
print("FINAL CHURN LABEL INTEGRITY CHECK\n")

final_churn_rate = drivers_df["is_churned"].mean() * 100
initial_rate     = 40.30  # from data generator output

print(f"  Initial churn rate  : {initial_rate:.2f}%")
print(f"  After cleaning      : {final_churn_rate:.3f}%")
print(f"  Drift               : "
      f"{final_churn_rate - initial_rate:+.4f}%")

if abs(final_churn_rate - initial_rate) < 0.5:
    print(f"  ✅ Churn rate stable")
else:
    print(f"  ⚠️ Churn rate shifted significantly — investigate")

# Verify weekly rows per driver (should be 16)
rows_per_driver = weekly_df.groupby("driver_id").size()
print(f"\n  Weekly rows per driver:")
print(f"    Min  : {rows_per_driver.min()}")
print(f"    Max  : {rows_per_driver.max()}")
print(f"    Mean : {rows_per_driver.mean():.2f}")
print(f"    = 16 for all drivers: "
      f"{'✅ Yes' if rows_per_driver.min() == 16 else '⚠️ No'}")

# Verify no nulls in critical columns
critical_cols = [
    "rides_this_week","weekly_earnings",
    "cancellation_rate","hours_online","login_days"
]
print(f"\n  Nulls in critical weekly columns:")
for col in critical_cols:
    n = weekly_df[col].isnull().sum()
    status = "✅" if n == 0 else "⚠️"
    print(f"    {col:<25}: {n:>4} {status}")

print(f"\n  Nulls in driver modeling columns:")
model_driver_cols = [
    "is_churned","rating","age","days_on_platform","zones_known"
]
for col in model_driver_cols:
    n = drivers_df[col].isnull().sum()
    status = "✅" if n == 0 else "⚠️"
    print(f"    {col:<25}: {n:>4} {status}")

FINAL CHURN LABEL INTEGRITY CHECK

  Initial churn rate  : 40.30%
  After cleaning      : 40.300%
  Drift               : +0.0000%
  ✅ Churn rate stable

  Weekly rows per driver:
    Min  : 16
    Max  : 16
    Mean : 16.00
    = 16 for all drivers: ✅ Yes

  Nulls in critical weekly columns:
    rides_this_week          :    0 ✅
    weekly_earnings          :    0 ✅
    cancellation_rate        :    0 ✅
    hours_online             :    0 ✅
    login_days               :    0 ✅

  Nulls in driver modeling columns:
    is_churned               :    0 ✅
    rating                   :    0 ✅
    age                      :    0 ✅
    days_on_platform         :    0 ✅
    zones_known              :    0 ✅


In [11]:
drivers_df.to_csv(   DATA_PROCESSED + "drivers_clean.csv",
                     index=False)
weekly_df.to_csv(    DATA_PROCESSED + "weekly_activity_clean.csv",
                     index=False)
incentive_df.to_csv( DATA_PROCESSED + "incentives_clean.csv",
                     index=False)
tickets_df.to_csv(   DATA_PROCESSED + "support_tickets_clean.csv",
                     index=False)

print("✅ All cleaned tables saved to data/processed/")
print(f"  drivers_clean.csv          : "
      f"{len(drivers_df):,} rows × {len(drivers_df.columns)} columns")
print(f"  weekly_activity_clean.csv  : "
      f"{len(weekly_df):,} rows × {len(weekly_df.columns)} columns")
print(f"  incentives_clean.csv       : "
      f"{len(incentive_df):,} rows × {len(incentive_df.columns)} columns")
print(f"  support_tickets_clean.csv  : "
      f"{len(tickets_df):,} rows × {len(tickets_df.columns)} columns")

✅ All cleaned tables saved to data/processed/
  drivers_clean.csv          : 5,000 rows × 20 columns
  weekly_activity_clean.csv  : 80,000 rows × 17 columns
  incentives_clean.csv       : 7,283 rows × 6 columns
  support_tickets_clean.csv  : 10,028 rows × 6 columns


## Cleaning Summary

### What We Fixed

| Step | Action | Why |
|---|---|---|
| Duplicates | Composite key (driver_id + week_number) dedup | Each driver should have exactly one row per week |
| Data types | Timestamps, booleans, numerics per table | Enable correct calculations |
| Null audit | Classified each null as conditional/structural/quality | Core principle — not all nulls are problems |
| Structural zeros | incentive_amount, complaints, zones → 0 for inactive weeks | Inactive weeks genuinely contribute zero |
| Forward-fill | weekly_rating within driver series | Rating persists across inactive weeks — 14,746 nulls handled |
| Quality nulls | Median by vehicle_type for earnings, median for others | Genuine missing data — 4,796 nulls across 3 columns |
| Corrupt ages | Outside 18-70 → median | Domain constraint |
| Outlier capping | 1st/99th percentile weekly, IQR for driver cols | Preserve trend signals while removing artifacts |
| Label integrity | Churn rate monitored after every step | Cannot corrupt the target variable |

---

### Null Audit Findings — What Each Category Told Us

**weekly_rating (18.4% null → ✅ Conditionally expected):**
83.6% of these nulls coincide with zero-ride weeks —
exactly as expected. A driver who did not work has no
rating for that week. Forward-fill within the driver's
own time series was the correct strategy. This preserved
the temporal continuity of each driver's reputation score,
which will be important when we compute "rating trend
over 12 weeks" in Feature Engineering.

**hours_online, cancellation_rate, weekly_earnings
(1.5-2.6% null → ⚠️ Data quality nulls):**
Only 14-16% of these nulls coincide with zero-ride weeks
— meaning most nulls exist in weeks where the driver WAS
active. These are genuine data collection failures, not
expected missing values. Correctly identified and imputed
with medians stratified by vehicle type (earnings) or
activity status (hours, cancellation rate).

**churn_reason (59.7% null → ⚠️ Flagged as data quality
but actually STRUCTURAL):**
The null audit flagged this as a data quality null because
zero-ride weeks had 0% null rate for this column (the audit
checked weekly rows, not driver rows). But churn_reason
lives in the weekly_activity table as a driver-level
attribute — null means "this driver did not churn."
59.7% null is CORRECT — 59.7% of drivers are retained.

This is an important lesson: automated null audits are
a starting point, not the final answer. Domain knowledge
must always validate what the automation reports.
We will NOT use churn_reason as a model feature because
it is only known for churned drivers (information available
after the fact, not at prediction time). This would be
a severe data leakage issue.

---

### Integrity Checks — All Passed

| Check | Result |
|---|---|
| Churn rate drift | +0.0000% — perfectly stable |
| Weekly rows per driver | Min=16, Max=16, Mean=16.00 — perfect |
| Critical weekly nulls | All zero after cleaning |
| Driver modeling nulls | All zero after cleaning |

**Every driver has exactly 16 weekly rows** — this is a critical
structural requirement for Feature Engineering. If any driver
had 14 or 17 rows, our "last 4 weeks vs first 8 weeks"
trend calculations would be misaligned. The perfect
Min=16, Max=16 result confirms the composite key deduplication
worked correctly and the time series is structurally intact.

**Zero drift in churn rate (+0.0000%)** — this is the strongest
possible signal that our cleaning was balanced. No cleaning
step accidentally removed or altered churn labels
disproportionately from either class.

---

### The One Column We Deliberately Left Dirty

**`churn_reason`** — not cleaned, not imputed, not used as a feature.

This column tells us WHY a driver churned (earnings_drop,
competitor_platform, vehicle_issue, etc.) — but only for
churned drivers. Retained drivers have null. Using it as
a feature would mean the model sees "this driver had
churn_reason = earnings_drop" and immediately knows
they churned — perfect prediction, zero learning.
This is the most obvious form of data leakage possible.

We preserve it in the cleaned table for analysis purposes
only — it will be excluded from Feature Engineering.
In a real Ola deployment, churn reason would only be
collected via exit surveys after churn has already
occurred — it would never be available at prediction time.

---

### New Concepts Introduced in This Notebook

| Concept | Previous Projects | This Project |
|---|---|---|
| Null classification | Quality vs structural | Quality vs structural vs **conditional** |
| Fill strategy | Global median | **Forward-fill within entity time series** |
| Outlier capping | IQR (1.5x) | **Gentle 1/99 percentile (preserve trends)** |
| Deduplication key | Single primary key | **Composite key (driver_id + week_number)** |
| Label monitoring | Once at start | **After every cleaning step** |
| Leakage from null | Not applicable | **churn_reason = post-hoc information** |

---

### What We Have Now — Ready for EDA

Four clean tables with full structural integrity:

| Table | Rows | Key Property |
|---|---|---|
| drivers_clean.csv | 5,000 | One row per driver, churn label attached |
| weekly_activity_clean.csv | 80,000 | 16 rows per driver, no critical nulls |
| incentives_clean.csv | ~7,300 | One row per offer, all values valid |
| support_tickets_clean.csv | ~10,000 | Pending nulls preserved intentionally |

---

### Next Step → Notebook 03 — EDA

We ask two questions:

**Question 1 (Driver level):** What profile characteristics —
experience, vehicle type, rating, income dependency —
correlate most strongly with churn?

**Question 2 (Behavioral trajectory):** Which weekly
behavioral patterns — rides declining, earnings falling,
cancellation rising — most strongly predict churn?

Question 2 is the one we have not been able to ask in
any previous project. It is the heart of churn prediction.